In [1]:
import torch
import sys

sys.path.append("/home/atuin/v120bb/v120bb18/UnReflectAnything")
from utilities.visualization import rgb, panelize
from polar_highlighter import PolarHighlighter

if torch.cuda.is_available():
    num_devices = torch.cuda.device_count()
    curr_device = torch.cuda.current_device()
    device_name = torch.cuda.get_device_name(curr_device)
    print(f"CUDA is available: {num_devices} device(s) detected.")
    print(f"Current device id: {curr_device} - {device_name}")
else:
    print("CUDA is not available")
%load_ext autoreload
%autoreload 2


CUDA is available: 1 device(s) detected.
Current device id: 0 - NVIDIA A40


In [2]:
from main import load_and_process_config
from dataset import from_config
from utilities import tensor_dict_summarize
from tqdm.notebook import tqdm

config = load_and_process_config("../config_train.yaml")
config_scrream = config
# config_scrream.DATASETS = {"CROMO": config.DATASETS.CROMO}
dataset = from_config(config_scrream)["training"]
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False,num_workers=6,prefetch_factor=6)
# highlighter = PolarHighlighter(width=448, height=448).cuda()

sizes = []
maxs = []
mins = []
for batch in tqdm(dataloader):
    # batch = {
    #     k: v.cuda() if isinstance(v, torch.Tensor) and torch.cuda.is_available() else v
    #     for k, v in batch.items()
    # }
    sizes.append(batch["diffuse"].shape[2])
    maxs.append(batch["specular"].max())
    mins.append(batch["diffuse"].min())
    break


FileNotFoundError: [Errno 2] No such file or directory: '../config_train.yaml'

In [13]:
def dimensions(training_dl, config):
    """Extract dimensions from the training data"""
    # Extract frame dimensions
    input_shape = next(iter(training_dl))[
        "raw"
    ].shape  # [batch_size, channels, height, width]
    sample_shape = next(iter(training_dl))["raw"].shape[1:]  # [channels, height, width]
    height = sample_shape[-2]  # Image height
    width = sample_shape[-1]  # Image width
    channels = 3  # Number of image channels
    batch_size = next(iter(training_dl))["raw"].shape[0]  # Batch size

    return {
        "input_shape": input_shape,
        "sample_shape": sample_shape,
        "height": height,
        "width": width,
        "channels": channels,
        "batch_size": batch_size,
    }
    
dimensions(dataloader, config)

{'input_shape': torch.Size([1, 3, 896, 896]),
 'sample_shape': torch.Size([3, 896, 896]),
 'height': 896,
 'width': 896,
 'channels': 3,
 'batch_size': 1}

In [ ]:
from polar_highlighter import get_soft_highlight_map

real_highlight_soft_mask = get_soft_highlight_map(
    batch["diffuse"].to("cuda", non_blocking=True),
    threshold=config.SOFT_HIGHLIGHT_THRESHOLD,
)
# Compute inverse binary mask to mask out real highlights from the loss computation
real_highlight_inverse_binary_mask = torch.logical_not(
    torch.nn.functional.max_pool2d(
        real_highlight_soft_mask,
        kernel_size=config.REAL_HIGHLIGHT_DILATION,
        stride=1,
        padding=config.REAL_HIGHLIGHT_DILATION // 2,
    )
    > 0
).int()
rgb(real_highlight_inverse_binary_mask * batch["diffuse"])

In [ ]:
real_highlight_inverse_binary_mask.shape

In [ ]:
from loss_utils import hole_and_ring_masks

hole_mask, ring_mask = hole_and_ring_masks(real_highlight_inverse_binary_mask, 21)

In [ ]:
rgb(hole_mask * batch["diffuse"])
rgb(ring_mask * batch["diffuse"])